In [1]:
import requests
from bs4 import BeautifulSoup
import csv

# Fungsi untuk konversi rating dari class CSS
def get_rating(star_class):
    ratings = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    for key in ratings:
        if key in star_class:
            return ratings[key]
    return None

# Fungsi untuk ambil kategori dari halaman detail
def get_category(book_url):
    res = requests.get(book_url)
    soup = BeautifulSoup(res.text, "html.parser")
    breadcrumb = soup.select("ul.breadcrumb li a")
    if len(breadcrumb) >= 3:
        return breadcrumb[2].text.strip()
    return "Unknown"

# Inisialisasi CSV
with open("books.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Judul", "Harga", "Ketersediaan", "Rating", "Kategori"])

    # Loop semua halaman
    for page in range(1, 51):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.find_all("article", class_="product_pod")

        for book in books:
            title = book.h3.a["title"]
            price = book.find("p", class_="price_color").text
            availability = book.find("p", class_="instock availability").text.strip()
            rating_class = book.find("p", class_="star-rating")["class"]
            rating = get_rating(rating_class)
            relative_url = book.h3.a["href"]
            book_url = "https://books.toscrape.com/catalogue/" + relative_url
            category = get_category(book_url)

            writer.writerow([title, price, availability, rating, category])
